In [ ]:
import seaborn as sns
import pandas as pd
import os

# Chargement initial
titanic = sns.load_dataset("titanic")
print(f"Lignes initiales : {len(titanic)}")

# 1. Traitement des valeurs manquantes
# A. deck : suppression de la colonne (77% de valeurs manquantes)
titanic_propre = titanic.drop(columns=["deck"])

# B. embark_town : suppression des 2 lignes sans port
titanic_propre = titanic_propre.dropna(subset=["embark_town"])

# C. age : imputation par la médiane de sa classe et de son sexe
mediane_age = titanic_propre.groupby(["class", "sex"])["age"].transform("median")
titanic_propre["age"] = titanic_propre["age"].fillna(mediane_age)

print(f"Manquants restants : {titanic_propre.isna().sum().sum()}")

# 2. Examen des doublons
nb_doublons = titanic_propre.duplicated().sum()
print(f"Lignes avec profil identique : {nb_doublons}")
# Décision : on conserve ces lignes car sans identifiant unique (nom/billet), ce sont des passagers réels distincts.

# 3. Suppression des colonnes redondantes et typage
titanic_propre = titanic_propre.drop(columns=["alive", "embarked"])

# 4. Détection des billets hors bornes (règle des 1.5 IQR)
q1 = titanic_propre["fare"].quantile(0.25)
q3 = titanic_propre["fare"].quantile(0.75)
iqr = q3 - q1
borne_haute = q3 + 1.5 * iqr
billets_hors_bornes = len(titanic_propre[titanic_propre["fare"] > borne_haute])
print(f"Billets supérieurs à la borne haute ({borne_haute:.2f} $) : {billets_hors_bornes}")
# Décision : valeurs rares mais réelles (suites de luxe) -> on les conserve.

# 5. Enregistrement dans data/titanic_propre.csv
os.makedirs("data", exist_ok=True)
titanic_propre.to_csv("data/titanic_propre.csv", index=False)

print(f"\nDimensions finales : {titanic_propre.shape}")
print("Fichier data/titanic_propre.csv enregistré avec succès !")
